# REMX volatility-model pipeline

Runs the model-estimation notebooks in sequence and stitches their captured outputs into a single combined report at `REPORTS/main.txt`.

1. `03-GP-models-gaussian.ipynb` — five G&P (2019) GARCH variants with Gaussian innovations
2. `03-GP-models-student.ipynb` — same five GARCH variants with standardized Student-t innovations
3. `04-extension-models.ipynb` — Sentiment-augmented EGARCH(1,1)-X under both Gaussian and Student-t innovations

In [3]:
import os
import nbformat
from nbclient import NotebookClient
from pathlib import Path

# Resolve the CODE/ directory robustly. The kernel's cwd may have been
# invalidated (FileNotFoundError from os.getcwd()) by a stale chdir from a
# previous run, so we try multiple sources in order of reliability.
def _resolve_code_dir() -> Path:
    # 1. VS Code Jupyter sets this to the absolute path of the running notebook
    vsc = globals().get('__vsc_ipynb_file__')
    if vsc:
        p = Path(vsc).resolve().parent
        if (p / 'main.ipynb').exists():
            return p
    # 2. Current working directory (may raise FileNotFoundError if stale)
    try:
        cwd = Path(os.getcwd())
        if (cwd / 'main.ipynb').exists():
            return cwd
        if (cwd / 'CODE' / 'main.ipynb').exists():
            return cwd / 'CODE'
    except FileNotFoundError:
        pass
    raise RuntimeError(
        'Cannot locate CODE/ directory. Restart the kernel and re-run, or '
        'set CODE_DIR manually before this cell.'
    )

CODE_DIR = _resolve_code_dir()
os.chdir(CODE_DIR)  # so the rest of the notebook also has a sane cwd
print(f'CODE_DIR = {CODE_DIR}')

NOTEBOOKS = [
    '03-GP-models-gaussian.ipynb',
    '03-GP-models-student.ipynb',
    '04-extension-models.ipynb',
]

for nb_name in NOTEBOOKS:
    nb_path = CODE_DIR / nb_name
    print(f'Executing {nb_path} ...')
    nb = nbformat.read(str(nb_path), as_version=4)
    client = NotebookClient(
        nb, timeout=600, kernel_name='python3',
        resources={'metadata': {'path': str(CODE_DIR)}},
    )
    client.execute()
    nbformat.write(nb, str(nb_path))
    print('  done.')

CODE_DIR = /Users/aleix/Documents/Repos/master_thesis_repo/CODE
Executing /Users/aleix/Documents/Repos/master_thesis_repo/CODE/03-GP-models-gaussian.ipynb ...
  done.
Executing /Users/aleix/Documents/Repos/master_thesis_repo/CODE/03-GP-models-student.ipynb ...
  done.
Executing /Users/aleix/Documents/Repos/master_thesis_repo/CODE/04-extension-models.ipynb ...
  done.


## Combined report

Reads each notebook's persisted capture and stitches them into `REPORTS/garch_report.txt` with one section per model.


In [4]:
from pathlib import Path
from datetime import datetime

# This notebook lives in CODE/, the consolidated reports live at the repo
# root in REPORTS/, so all paths are written as "../REPORTS".
REPORTS_DIR = Path("../REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)
OUT_PATH = REPORTS_DIR / "main.txt"

sections = [
    ("GARCH family — Gaussian innovations",        REPORTS_DIR / "03-GP-models-gaussian.txt"),
    ("GARCH family — Student-t innovations",       REPORTS_DIR / "03-GP-models-student.txt"),
    ("EGARCH(1,1)-X — Gaussian innovations",       REPORTS_DIR / "04-extension-models-gaussian.txt"),
    ("EGARCH(1,1)-X — Student-t innovations",      REPORTS_DIR / "04-extension-models-student.txt"),
]

def _banner(title, char='='):
    bar = char * 78
    return f"{bar}\n{title}\n{bar}\n"

with OUT_PATH.open("w") as f:
    f.write(_banner(f"REMX volatility-model estimation report  "
                    f"(generated {datetime.now():%Y-%m-%d %H:%M})"))
    f.write("\n")
    for title, cap_path in sections:
        block = _banner(title, '-')
        body  = cap_path.read_text() if cap_path.exists() else "(capture file not found - re-run the source notebook)\n"
        f.write(block)
        f.write(body)
        f.write("\n\n")
        print(block + body)

print(f"Saved -> {OUT_PATH}  ({OUT_PATH.stat().st_size:,} bytes)")


------------------------------------------------------------------------------
GARCH family — Gaussian innovations
------------------------------------------------------------------------------
------------------------------------------------------------------------------
GARCH(1,1) -- Gaussian baseline
------------------------------------------------------------------------------
                       Zero Mean - GARCH Model Results                        
Dep. Variable:                      r   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -5982.31
Distribution:                  Normal   AIC:                           11970.6
Method:            Maximum Likelihood   BIC:                           11988.4
                                        No. Observations:                 2765
Date:                Sat, May 30 2026   Df Residuals:          